## Filtering for artefacts
## Calculating percentages of bridges, tunnels, viaducts and lowerlying areas
## aggregating to netwekschakels
## on/off ramp analysis

In [ ]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, Polygon, box
import rasterio
import os
import matplotlib.pyplot as plt
import folium
from branca.colormap import LinearColormap
import os
import numpy as np
from shapely.ops import transform
import pyproj

In [ ]:
region_list = ["ARK-NZK","Vallei en Veluwe",
               "Achterhoek", "Brabantse Delta","Friesland",
               "Groningen en NO-Drenthe","Limburg",
               "Noord-Brabant Oost","Noord-Westelijke Delta",
               "Rivierenland","Scheldestromen","Zuiderzeeland",
               "Overijsselse Vecht"
               ]

In [8]:
region_list = ["Overijsselse Vecht"]

In [ ]:
region_list = [
                "ARK-NZK","Vallei en Veluwe",
                "Brabantse Delta","Friesland",
               "Groningen en NO-Drenthe","Limburg",
               "Noord-Brabant Oost","Noord-Westelijke Delta",
               "Rivierenland","Scheldestromen","Zuiderzeeland",
               "Overijsselse Vecht"
               ]

In [ ]:
# Assume lower_lying_regions contains both line and polygon layers
'''
from pathlib import Path
import geopandas as gpd
import pandas as pd

lowerlying_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Lowerlying_sections")
lowerlying_gpkgs = list(lowerlying_dir.glob("*.gpkg"))
print(f"Found {len(lowerlying_gpkgs)} lowerlying GeoPackages.")

# Read all GeoPackages into a list of GeoDataFrames
lower_lying_regions = [gpd.read_file(gpkg) for gpkg in lowerlying_gpkgs]

buffer_distance = 1  # meters

buffered_gdfs = []
polygon_gdfs = []

for gdf in lower_lying_regions:
    geom_types = set(gdf.geometry.geom_type)
    if 'LineString' in geom_types or 'MultiLineString' in geom_types:
        # Buffer lines to polygons
        gdf_buffered = gdf.copy()
        gdf_buffered['geometry'] = gdf_buffered.geometry.buffer(buffer_distance)
        buffered_gdfs.append(gdf_buffered)
    else:
        polygon_gdfs.append(gdf)

# Combine all as polygons
all_polygons = buffered_gdfs + polygon_gdfs

# Ensure all have the same CRS
target_crs = all_polygons[0].crs
all_polygons = [gdf.to_crs(target_crs) for gdf in all_polygons]

# Merge into one GeoDataFrame
merged_lower_lying = gpd.GeoDataFrame(
    pd.concat(all_polygons, ignore_index=True),
    crs=target_crs
)

output_path = lowerlying_dir / "processed" / "merged_lower_lying_buffered.gpkg"
if output_path.exists():
    output_path.unlink()
merged_lower_lying.to_file(output_path, driver="GPKG")
print(f"Merged (buffered) GeoDataFrame written to: {output_path}")
'''

In [ ]:
#fixing complications in the road -> should we fix these? -> I think these are weird for losses

region_list = [
               
               "Groningen en NO-Drenthe",
               "Scheldestromen",
               "Overijsselse Vecht"
               ]


#033-1030-L
#035-0002-L
#

In [ ]:
lowerlying_dir_edited = Path(r"P:\bovenregionale-stresstest-hwn\Data\Lowerlying_sections\processed\merged_lower_lying_edited.gpkg")
merged_lower_lying_edited = gpd.read_file(lowerlying_dir_edited)

merged_lower_lying_edited = gpd.read_file(lowerlying_dir_edited)

# Buffer only the sides with 2m (excluding the ends for LineString geometries)
buffer_distance = 4  # meters

def buffer_sides_only(geometry):
    """Buffer only the sides of geometries, excluding endpoints for LineStrings"""
    if geometry.geom_type == 'LineString':
        # For LineStrings, buffer normally but this will include ends
        # To exclude ends, we can use a cap_style parameter
        return geometry.buffer(buffer_distance, cap_style=2)  # cap_style=2 is flat/square
    elif geometry.geom_type == 'MultiLineString':
        # Apply the same logic to each LineString in the MultiLineString
        from shapely.geometry import MultiPolygon
        buffered_parts = [line.buffer(buffer_distance, cap_style=2) for line in geometry.geoms]
        return MultiPolygon(buffered_parts) if len(buffered_parts) > 1 else buffered_parts[0]
    else:
        # For other geometry types (Polygon, etc.), use regular buffer
        return geometry.buffer(buffer_distance)

# Apply side-only buffering
merged_lower_lying_edited['geometry'] = merged_lower_lying_edited['geometry'].apply(buffer_sides_only)

In [ ]:
from post_processing_functions import Thresholding_for_artefacts,Thresholding_for_artefacts,Filter_and_aggregate_flooded_segments_exposure, Filter_and_aggregate_flooded_segments_damage, calculate_overlay_percentages,get_z_height_optimized
import pandas as pd 
# add tunnels and bridge % columns to exposure and damage files and filter

data_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Processed_data")


tunnels = data_dir.joinpath("Tunnels_filtered_by_area_2000_th.gpkg")
bridges = data_dir.joinpath("filtered_bridges.gpkg")
kunstinweg = data_dir.joinpath("kunstinweg.shp")
road_height_path = data_dir.joinpath("road_height_points.gpkg")

kunstinweg_gdf = gpd.read_file(kunstinweg)
kunstinweg_gdf['geometry'] = kunstinweg_gdf['geometry'].buffer(0.2) # to make sure lines are valid
kunstinweg_gdf.rename(columns={'OMSCHR': 'objecttekst'}, inplace=True)

kunstinweg_bridge = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'brug']
kunstinweg_tunnel = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'tunnel']


tunnels_gdf_kunstoverweg = gpd.read_file(tunnels)
bridges_gdf_kunstoverweg = gpd.read_file(bridges) 

bridges_gdf = gpd.GeoDataFrame(
    pd.concat([bridges_gdf_kunstoverweg, kunstinweg_bridge], ignore_index=True),
    crs=bridges_gdf_kunstoverweg.crs
)

tunnels_gdf = gpd.GeoDataFrame(
    pd.concat([tunnels_gdf_kunstoverweg, kunstinweg_tunnel], ignore_index=True),
    crs=tunnels_gdf_kunstoverweg.crs
)



# Define allowed values for bridges and tunnels (lowercased for case-insensitive matching)
allowed_bridges = [
    'aanbrug', 'brug', 'brug (beweegbaar)', 'brug (landbouw)', 'brug (vast)',
    'brug beton', 'brug beton in', 'brug beton over', 'brug beweegbaar',
    'brug hout in', 'brug in', 'brug in de toerit va', 'brug staal in',
    'brug vast', 'vaste brug'
]

allowed_tunnels = [
    'cervedict tunnel', 'open tunnelbak', 'tunnel', 'tunnel vlak',
    'tunnelbak', 'tunnelbak den kaat'
]

filtered_viaducts = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'viaduct']

# Convert to lowercase for case-insensitive comparison
allowed_bridges = [x.lower() for x in allowed_bridges]
allowed_tunnels = [x.lower() for x in allowed_tunnels]

# Filter bridges
filtered_gdf_brug = bridges_gdf[
    bridges_gdf['objecttekst'].str.lower().isin(allowed_bridges)
]

# Filter tunnels
filtered_gdf_tunnel_and_bridges = tunnels_gdf[
    tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels + allowed_bridges)
]

'''
tunnel_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Tunnels_Bridges\Tunnels.gpkg")
filtered_gdf_tunnel_and_bridges.to_file(tunnel_path, driver="GPKG")

brug_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Tunnels_Bridges\Bridges.gpkg")
filtered_gdf_brug.to_file(brug_path, driver="GPKG")

viaducts_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Tunnels_Bridges\Viaducts.gpkg")
filtered_viaducts.to_file(viaducts_path, driver="GPKG")
'''


In [ ]:
# Add EV2_me values for provinces

#F_EV2_me
no_duration_48_uur = ['Limburg']
no_duration_240_uur = ["Friesland","Groningen en NO-Drenthe"]
for region in region_list:
    print(f"Processing region: {region} network")
    if region in no_duration_48_uur:
        print(f"Applying 48 uur specific processing for region: {region}")
        root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
        roads_ex = root_dir / "damages/ML_damage_segmented.gpkg"
        roads_ex_gdf = gpd.read_file(roads_ex)

        roads_ex_gdf['F_EV2_ma'] = 48

        # Replace the original file
        roads_ex_gdf.to_file(roads_ex, driver='GPKG')
        print(f"Updated {roads_ex} with F_EV2_ma = 48") 
        
        pass
    elif region in no_duration_240_uur:
        print(f"Applying 240 uur specific processing for region: {region}")
        root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
        roads_ex = root_dir / "damages/ML_damage_segmented.gpkg"
        roads_ex_gdf = gpd.read_file(roads_ex)

        roads_ex_gdf['F_EV2_ma'] = 240

        roads_ex_gdf.to_file(roads_ex, driver='GPKG')
        print(f"Updated {roads_ex} with F_EV2_ma = 240")
        
        pass


In [10]:
from post_processing_functions import Thresholding_for_artefacts,Aggregate_flooded_segments, calculate_overlay_percentages,get_z_height_optimized


for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "damages/ML_damage_segmented.gpkg"
    
    roads_ex_gdf = gpd.read_file(roads_ex)
    
    points_gdf = gpd.read_file(road_height_path)
    points_gdf = points_gdf.to_crs(roads_ex_gdf.crs)
    roads_ex_gdf['Z_height'] = get_z_height_optimized(roads_ex_gdf, points_gdf, threshold=50.0)
    
    print("Calculating tunnel and bridge percentages...")
    Roads_assets = calculate_overlay_percentages(roads_ex_gdf, filtered_gdf_brug, filtered_gdf_tunnel_and_bridges,filtered_viaducts,merged_lower_lying_edited)
    print("Applying thresholding to remove artefacts...")
    dataframe = Thresholding_for_artefacts("F_","Damages", Roads_assets, root_dir)
    print(dataframe.columns)
    print("Filtering and aggregating flooded segments...")
    Aggregate_flooded_segments(dataframe, root_dir,"Aggregated", dissolve_col='NETWERKSCH_HWN')
    


Processing region: Overijsselse Vecht network
Calculating tunnel and bridge percentages...
Applying thresholding to remove artefacts...


CPLE_AppDefinedError: b'sqlite3_exec(CREATE TRIGGER "trigger_delete_feature_count_Damages_Filtering_all_columns" AFTER DELETE ON "Damages_Filtering_all_columns" BEGIN UPDATE gpkg_ogr_contents SET feature_count = feature_count - 1 WHERE lower(table_name) = lower(\'Damages_Filtering_all_columns\'); END;) failed: unable to open database file'

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b'sqlite3_exec(CREATE TRIGGER "trigger_delete_feature_count_Damages_Filtering_all_columns" AFTER DELETE ON "Damages_Filtering_all_columns" BEGIN UPDATE gpkg_ogr_contents SET feature_count = feature_count - 1 WHERE lower(table_name) = lower(\'Damages_Filtering_all_columns\'); END;) failed: unable to open database file'


Index(['link_id', 'REF_ID', 'infra_type', 'avgspeed', 'lanes', 'id_NWB',
       'BST_CODE_NWB', 'WVK_ID', 'FOW_NWB', 'HECTO_LTTR_NWB', 'NETWERKSCH_HWN',
       'NWBWGNR_HWN', 'NWBWGDL_HWN', 'node_A', 'node_B', 'edge_fid', 'rfid_c',
       'rfid', 'length', 'time', 'F_EV1_mi', 'F_EV1_ma', 'F_EV1_me',
       'F_EV1_fr', 'F_EV2_mi', 'F_EV2_ma', 'F_EV2_me', 'F_EV2_fr', 'road_type',
       'lanes_copy', 'dam_EV1_al', 'dam_EV2_al', 'geometry', 'Z_height',
       'tunnel_percentage', 'bridge_percentage', 'viaduct_percentage',
       'lowerlying_percentage', 'OR_me_10_fr_25', 'AND_me_10_fr_20', 'culvert',
       'me_20', 'isolated', 'is_flooded', 'Asset', 'flooded_bridge',
       'artefact', 'remove', 'is_isolated', 'flooded_tunnel_entrance',
       'flooded_lowerlying_entrance'],
      dtype='object')
Filtering and aggregating flooded segments...


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels')) failed: disk I/O error"


In [11]:
#On/OFF ramps with Losses network -> Because its dissolved :)
from post_processing_functions import cluster_connected,aggregate_clusters_to_points
root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")

losses_network = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Losses_Analysis\Network_for_complex_losses.gpkg")
network_gdf = gpd.read_file(losses_network)
ramps_gdf = network_gdf[network_gdf["BST_CODE_NWB"].isin(["AFR", "OPR"])]
afr_gdf = ramps_gdf[ramps_gdf["BST_CODE_NWB"] == "AFR"].copy()
opr_gdf = ramps_gdf[ramps_gdf["BST_CODE_NWB"] == "OPR"].copy()
afr_gdf_clustered = cluster_connected(afr_gdf)

afr_gdf_aggregated = aggregate_clusters_to_points(afr_gdf_clustered, "F_EV1_me", method="max")
output_gpkg_aggregated_afr = root_dir / "Country_scale_afr_Points.gpkg"
afr_gdf_aggregated.to_file(output_gpkg_aggregated_afr, driver="GPKG")

opr_gdf_clustered = cluster_connected(opr_gdf)
opr_gdf_aggregated = aggregate_clusters_to_points(opr_gdf_clustered, "F_EV1_me", method="max")
output_gpkg_aggregated = root_dir / "Country_scale_opr_Points.gpkg"
opr_gdf_aggregated.to_file(output_gpkg_aggregated, driver="GPKG")

In [ ]:
#On off ramp analysis
from pathlib import Path
import geopandas as gpd
from post_processing_functions import cluster_connected,aggregate_clusters_to_points



for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    network_file = root_dir / "Damages_Filtering_all_columns.gpkg"
    network_gdf = gpd.read_file(network_file)

    ramps_gdf = network_gdf[network_gdf["BST_CODE_NWB"].isin(["AFR", "OPR"])]


    afr_gdf = ramps_gdf[ramps_gdf["BST_CODE_NWB"] == "AFR"].copy()
    opr_gdf = ramps_gdf[ramps_gdf["BST_CODE_NWB"] == "OPR"].copy()

   

    afr_gdf_clustered = cluster_connected(afr_gdf)
    #output_gpkg = root_dir / "afr_LineSegments.gpkg"
    #afr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    
    afr_gdf_aggregated = aggregate_clusters_to_points(afr_gdf_clustered, "F_EV1_me", method="max")
    output_gpkg_aggregated_afr = root_dir / "afr_Points.gpkg"
    afr_gdf_aggregated.to_file(output_gpkg_aggregated_afr, driver="GPKG")

   

    opr_gdf_clustered = cluster_connected(opr_gdf)
    
    #output_gpkg = root_dir / "opr_LineSegments.gpkg"
    #opr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    opr_gdf_aggregated = aggregate_clusters_to_points(opr_gdf_clustered, "F_EV1_me", method="max")
    output_gpkg_aggregated = root_dir / "opr_Points.gpkg"
    opr_gdf_aggregated.to_file(output_gpkg_aggregated, driver="GPKG")
